# SQL Murder Mistery

![image-2.png](attachment:image-2.png)

![image-2.png](https://drive.google.com/uc?id=1QxBiB5dMkAkOB0aksGM3yOh1QiBq3He-)

Ha ocurrido un crimen y el detective necesita tu ayuda. El detective te dio el informe de la escena del crimen, pero de alguna manera lo perdiste. Recuerdas vagamente que el crimen fue un **asesinato** que ocurrió en algún momento del **15 de enero de 2018** y que tuvo lugar en **SQL City**. Comienza recuperando el informe de la escena del crimen correspondiente de la base de datos del departamento de policía.
Vamos a trabajar con la Base de datos del fichero: **sql-murder-mystery**

El diagrama de la Base de Datos es el siguiente:

![diagrama](attachment:image.png)

![diagrama](https://drive.google.com/uc?id=1s9oYP4U7nSQL66XnT5KZI17qwlFiGIQm)

### 1. Importamos la Base de Datos

In [ ]:
import sqlite3

conexion = sqlite3.connect('sql-murder-mystery.db')
cursor = conexion.cursor()

### 2. Extraemos los datos de la escena del crimen

In [ ]:
cursor.execute("SELECT * FROM crime_scene_report WHERE type = 'murder' AND date = '20180115' AND city = 'SQL City'")
crime_scene_report = cursor.fetchall()

print("Informe de la escena del crimen:")
for fila in crime_scene_report:
    print(fila)

Informe de la escena del crimen:
(20180115, 'murder', 'Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".', 'SQL City')


### 3 Datos de los Testigos

In [ ]:
# Testigo 1: La última casa en 'Northwestern Dr'
cursor.execute("SELECT * FROM person WHERE address_street_name = 'Northwestern Dr' ORDER BY address_number DESC LIMIT 1")
testigo1 = cursor.fetchall()

print("Primer testigo:")
for fila in testigo1:
    print(fila)

# Testigo 2: Annabel en 'Franklin Ave'
cursor.execute("SELECT * FROM person WHERE name LIKE 'Annabel%' AND address_street_name = 'Franklin Ave'")
testigo2 = cursor.fetchall()

print("Segundo testigo:")
for fila in testigo2:
    print(fila)

Primer testigo:
(14887, 'Morty Schapiro', 118009, 4919, 'Northwestern Dr', '111564949')
Segundo testigo:
(16371, 'Annabel Miller', 490173, 103, 'Franklin Ave', '318771143')


### 4. Datos interrogatorios Testigos

In [ ]:
# Interrogatorio del primer testigo
cursor.execute("SELECT * FROM interview WHERE person_id = 14887")
interrogatorioMorty = cursor.fetchall()

print("Interrogatorio de Morty Schapiro:")
for fila in interrogatorioMorty:
    print(fila)

# Interrogatorio del segundo testigo
cursor.execute("SELECT * FROM interview WHERE person_id = 16371")
interrogatorioAnnabel = cursor.fetchall()

print("\nInterrogatorio de Annabel Miller:")
for fila in interrogatorioAnnabel:
    print(fila)

Interrogatorio de Morty Schapiro:
(14887, 'I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".')

Interrogatorio de Annabel Miller:
(16371, 'I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.')


### 5. Buscamos a los Sospechosos que nos indican los testigos

In [ ]:
cursor.execute(
    """SELECT p.id, p.name, dl.plate_number, gfnm.id as membership_id
    FROM person p
    JOIN drivers_license dl ON p.license_id = dl.id
    JOIN get_fit_now_member gfnm ON p.id = gfnm.person_id
    JOIN get_fit_now_check_in gfnc ON gfnm.id = gfnc.membership_id
    WHERE gfnm.id LIKE '48Z%'
    AND gfnc.check_in_date = 20180109
    AND dl.plate_number LIKE '%H42W%'"""
)
sospechosos = cursor.fetchall()

print("Sospechosos:")
for sospechoso in sospechosos:
    print(sospechoso)

Sospechosos:
(67318, 'Jeremy Bowers', '0H42W2', '48Z55')


### 6. Veamos qué dijo el sospechoso en el interrogatorio.

* ¿Sabes quien ha cometido el crimen? Analiza su interrogatorio.

* ¿Tienes los nombres de todos los responsables?

In [ ]:
# Interrogatorio de Jeremy Bowers
cursor.execute("SELECT * FROM interview WHERE person_id = 67318")
interrogatorioJeremy = cursor.fetchall()

print("Interrogatorio de Jeremy Bowers:")
for fila in interrogatorioJeremy:
    print(fila)

Interrogatorio de Jeremy Bowers:
(67318, 'I was hired by a woman with a lot of money. I don\'t know her name but I know she\'s around 5\'5" (65") or 5\'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n')


### 7. Encuentra al Verdadero Culpable

**Jummmmmmm qué está pasando aquí!!**

Busca al verdadero culpable utilizando una única sentencia SELECT o utiliza funciones de orden superior para filtrar los datos.

In [ ]:
cursor.execute(
    """SELECT p.name, p.id, COUNT(fec.event_name) as concert_count
    FROM person p
    JOIN drivers_license dl ON p.license_id = dl.id
    JOIN facebook_event_checkin fec ON p.id = fec.person_id
    WHERE dl.gender = 'female'
      AND dl.height >= 65 AND dl.height <= 67
      AND dl.hair_color = 'red'
      AND dl.car_make = 'Tesla' AND dl.car_model = 'Model S'
      AND fec.event_name = 'SQL Symphony Concert'
      AND fec.date >= 20171201 AND fec.date <= 20171231
    GROUP BY p.id, p.name
    HAVING COUNT(fec.event_name) = 3;"""
)

culpable = cursor.fetchall()

print("Verdadero Culpable:")
for fila in culpable:
    print(fila)

Verdadero Culpable:
('Miranda Priestly', 99716, 3)


## 8. Haz tus Detenciones

* Pon aquí el resultado de tus investigaciones y quien/es deberían ser enchironados. Aporta todos los datos que consideres necesarios.


# **Pruebas definitivas:**


---



1.   **Miranda Priestly (ID: 99716)**:


---


Es el cerebro del crimen y quien contrató a Jeremy Bowers para cometer el asesinato.


  *   Mujer de pelo rojo, que mide entre 65 y 67 pulgadas de altura.
  *   Conduce un Tesla Model S.


*   Asistió al 'SQL Symphony Concert' exactamente 3 veces en diciembre de 2017.







---





2.  **Jeremy Bowers (ID: 67318):**


---

 Es el sicario contratado por Miranda Priestly.



*   Visto por un testigo huyendo de la escena con una bolsa de 'Get Fit Now Gym' con número de membresía '48Z'
*   Condujo un coche con matrícula que incluía 'H42W'


*   Su interrogatorio reveló que fue contratado como sicario por una mujer que coincide con Miranda Priestly.







